In [1]:
# Mount Drive Folder to Colab [Local]

from google.colab import drive
drive.mount("/content/gdrive")

!ln -s /content/gdrive/My\ Drive/ /mydrive
%cd /mydrive/LSMT-Stock-Predictor
!ls

Mounted at /content/gdrive
/content/gdrive/My Drive/LSMT-Stock-Predictor
LSTM-Stock-Predictor.ipynb  token.txt


In [2]:
# requirements

!pip3 install -q pyngrok flask flask-bootstrap yfinance numpy pandas scikit-learn tensorflow

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 456.4/456.4 kB 17.9 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Preparing metadata (setup.py) ... done


In [3]:
# Read ngrok token from file
ngrok_token = ''
with open("token.txt", "r") as f:
    for line in f:
        if line.startswith("NGROK_TOKEN"):
            ngrok_token = line.split("=")[1].strip()
            break

print("Token loaded:", ngrok_token[:4] + "...")  # show only first 4 chars for safety

Token loaded: 3D10...


In [4]:
import os

import numpy as np
import pandas as pd
import yfinance as yf
from sklearn.preprocessing import MinMaxScaler
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout
import warnings


from flask import Flask, render_template, request, jsonify
from pyngrok import ngrok
import threading

In [5]:
# CSS

CSS = """
<style>
@import url('https://fonts.googleapis.com/css2?family=DM+Mono:wght@300;400;500&family=Syne:wght@400;500;600;700&display=swap');

:root {
  --bg-base:       #0a0c0f;
  --bg-surface:    #111418;
  --bg-card:       #161b22;
  --bg-hover:      #1c2330;
  --border:        #21262d;
  --border-bright: #30363d;

  --accent-green:  #3fb950;
  --accent-blue:   #58a6ff;
  --accent-red:    #f85149;
  --accent-amber:  #d29922;

  --text-primary:  #e6edf3;
  --text-secondary:#8b949e;
  --text-muted:    #484f58;

  --radius-sm: 6px;
  --radius-md: 10px;
  --radius-lg: 14px;

  --font-display: 'Syne', sans-serif;
  --font-mono:    'DM Mono', monospace;
}

*, *::before, *::after { box-sizing: border-box; margin: 0; padding: 0; }

html { scroll-behavior: smooth; }

body {
  background-color: var(--bg-base);
  color: var(--text-primary);
  font-family: var(--font-mono);
  font-size: 14px;
  line-height: 1.6;
  min-height: 100vh;
  /* subtle grid texture */
  background-image:
    linear-gradient(rgba(48,54,61,0.15) 1px, transparent 1px),
    linear-gradient(90deg, rgba(48,54,61,0.15) 1px, transparent 1px);
  background-size: 32px 32px;
}

/* ── Layout ── */
.container {
  max-width: 860px;
  margin: 0 auto;
  padding: 2.5rem 1.5rem 4rem;
}

/* ── Header ── */
header {
  margin-bottom: 2.5rem;
  padding-bottom: 1.5rem;
  border-bottom: 1px solid var(--border);
  animation: fadeDown 0.5s ease both;
}

header h1 {
  font-family: var(--font-display);
  font-size: 2rem;
  font-weight: 700;
  letter-spacing: -0.02em;
  color: var(--text-primary);
  display: flex;
  align-items: center;
  gap: 0.5rem;
}

header p {
  margin-top: 0.35rem;
  font-size: 13px;
  color: var(--text-secondary);
  letter-spacing: 0.04em;
}

/* ── Controls ── */
.controls {
  display: flex;
  gap: 10px;
  margin-bottom: 2rem;
  flex-wrap: wrap;
  animation: fadeDown 0.5s 0.1s ease both;
}

.controls input,
.controls select {
  background: var(--bg-card);
  border: 1px solid var(--border-bright);
  color: var(--text-primary);
  font-family: var(--font-mono);
  font-size: 13px;
  padding: 0.6rem 0.875rem;
  border-radius: var(--radius-sm);
  outline: none;
  transition: border-color 0.2s, box-shadow 0.2s;
}

.controls input {
  flex: 1;
  min-width: 160px;
}

.controls input::placeholder {
  color: var(--text-muted);
}

.controls input:focus,
.controls select:focus {
  border-color: var(--accent-blue);
  box-shadow: 0 0 0 3px rgba(88,166,255,0.12);
}

.controls select {
  cursor: pointer;
  width: 110px;
}

.controls button {
  background: var(--accent-green);
  color: #0a0c0f;
  font-family: var(--font-display);
  font-size: 13px;
  font-weight: 600;
  padding: 0.6rem 1.25rem;
  border: none;
  border-radius: var(--radius-sm);
  cursor: pointer;
  letter-spacing: 0.03em;
  transition: background 0.2s, transform 0.1s, box-shadow 0.2s;
}

.controls button:hover {
  background: #56d364;
  box-shadow: 0 0 16px rgba(63,185,80,0.3);
}

.controls button:active {
  transform: scale(0.97);
}

/* ── Error ── */
.error {
  background: rgba(248,81,73,0.1);
  border: 1px solid rgba(248,81,73,0.3);
  color: var(--accent-red);
  padding: 0.75rem 1rem;
  border-radius: var(--radius-sm);
  font-size: 13px;
  margin-bottom: 1.25rem;
}

/* ── Loader ── */
.loader {
  color: var(--text-secondary);
  font-size: 13px;
  padding: 0.75rem 0;
  display: flex;
  align-items: center;
  gap: 10px;
  margin-bottom: 1.25rem;
}

.loader::before {
  content: '';
  display: inline-block;
  width: 14px;
  height: 14px;
  border: 2px solid var(--border-bright);
  border-top-color: var(--accent-green);
  border-radius: 50%;
  animation: spin 0.7s linear infinite;
}

/* ── Metrics Cards ── */
.metrics {
  display: grid;
  grid-template-columns: repeat(auto-fit, minmax(180px, 1fr));
  gap: 12px;
  margin-bottom: 1.75rem;
  animation: fadeUp 0.4s ease both;
}

.card {
  background: var(--bg-card);
  border: 1px solid var(--border);
  border-radius: var(--radius-md);
  padding: 1rem 1.25rem;
  display: flex;
  flex-direction: column;
  gap: 6px;
  transition: border-color 0.2s, background 0.2s;
}

.card:hover {
  border-color: var(--border-bright);
  background: var(--bg-hover);
}

.card span {
  font-size: 11px;
  color: var(--text-muted);
  text-transform: uppercase;
  letter-spacing: 0.08em;
}

.card strong {
  font-family: var(--font-display);
  font-size: 1.4rem;
  font-weight: 600;
  color: var(--text-primary);
  letter-spacing: -0.01em;
}

/* ── Chart ── */
#chart {
  display: block;
  width: 100% !important;
  background: var(--bg-card);
  border: 1px solid var(--border);
  border-radius: var(--radius-lg);
  padding: 1.25rem;
  margin-bottom: 1.75rem;
  animation: fadeUp 0.4s 0.1s ease both;
}

/* ── Forecast Grid ── */
.forecast-grid {
  display: grid;
  grid-template-columns: repeat(auto-fill, minmax(105px, 1fr));
  gap: 10px;
  animation: fadeUp 0.4s 0.2s ease both;
}

.forecast-card {
  background: var(--bg-card);
  border: 1px solid var(--border);
  border-radius: var(--radius-md);
  padding: 0.75rem 0.875rem;
  display: flex;
  flex-direction: column;
  gap: 4px;
  transition: border-color 0.2s, transform 0.15s;
}

.forecast-card:hover {
  border-color: var(--border-bright);
  transform: translateY(-2px);
}

.forecast-card .day {
  font-size: 10px;
  color: var(--text-muted);
  text-transform: uppercase;
  letter-spacing: 0.06em;
}

.forecast-card .price {
  font-family: var(--font-display);
  font-size: 1rem;
  font-weight: 600;
  color: var(--text-primary);
}

.forecast-card .diff {
  font-size: 11px;
  font-weight: 500;
}

.forecast-card .diff.up   { color: var(--accent-green); }
.forecast-card .diff.down { color: var(--accent-red); }

/* ── Utility ── */
.hidden { display: none !important; }

/* ── Animations ── */
@keyframes fadeDown {
  from { opacity: 0; transform: translateY(-12px); }
  to   { opacity: 1; transform: translateY(0); }
}

@keyframes fadeUp {
  from { opacity: 0; transform: translateY(10px); }
  to   { opacity: 1; transform: translateY(0); }
}

@keyframes spin {
  to { transform: rotate(360deg); }
}

/* ── Responsive ── */
@media (max-width: 600px) {
  header h1    { font-size: 1.5rem; }
  .controls    { flex-direction: column; }
  .controls input,
  .controls select,
  .controls button { width: 100%; }
  .forecast-grid { grid-template-columns: repeat(auto-fill, minmax(90px, 1fr)); }
}
</style>
"""

In [6]:
# JS

JS = """
<script src="https://cdn.jsdelivr.net/npm/chart.js"></script>
<script>
var API_BASE = "/api";
// const API_BASE = "http://localhost:5000/api";
var chartInstance = null;

async function predict() {
    const ticker = document.getElementById("ticker").value.trim().toUpperCase();
    const days = parseInt(document.getElementById("days").value);

    if (!ticker) return showError("Please enter a stock ticker.");

    showLoader(true);
    hideError();
    document.getElementById("results").classList.add("hidden");

    try {
        const res = await fetch(`${API_BASE}/predict?ticker=${ticker}&days=${days}`);
        const data = await res.json();

        if (!res.ok) throw new Error(data.error || "Prediction failed.");

        renderResults(data, days);
    } catch (err) {
        showError(err.message);
    } finally {
        showLoader(false);
    }
}

function renderResults(data, days) {
    document.getElementById("currentPrice").textContent = `$${data.current_price}`;
    document.getElementById("tickerLabel").textContent = data.ticker;
    const lastPred = data.predictions[data.predictions.length - 1];
    const change = (((lastPred - data.current_price) / data.current_price) * 100).toFixed(2);
    document.getElementById("targetPrice").textContent =
        `$${lastPred} (${change > 0 ? "+" : ""}${change}%)`;

    renderChart(data);
    renderForecastGrid(data.predictions, data.current_price);
    document.getElementById("results").classList.remove("hidden");
}

function renderChart(data) {
    const histLabels = data.history.map(d => d.Date);
    const histPrices = data.history.map(d => d.Close);

    const today = new Date();
    const predLabels = Array.from({ length: data.predictions.length }, (_, i) => {
        const d = new Date(today);
        d.setDate(d.getDate() + i + 1);
        return d.toISOString().split("T")[0];
    });

    const allLabels = [...histLabels, ...predLabels];
    const histData = [...histPrices, ...Array(data.predictions.length).fill(null)];
    const predData = [...Array(histPrices.length - 1).fill(null),
    histPrices[histPrices.length - 1], ...data.predictions];

    if (chartInstance) chartInstance.destroy();
    chartInstance = new Chart(document.getElementById("chart"), {
        type: "line",
        data: {
            labels: allLabels,
            datasets: [
                {
                    label: "Historical",
                    data: histData,
                    borderColor: "#3b82f6",
                    fill: false,
                    tension: 0.3,
                    pointRadius: 0,
                },
                {
                    label: "Predicted",
                    data: predData,
                    borderColor: "#10b981",
                    borderDash: [6, 4],
                    fill: false,
                    tension: 0.3,
                    pointRadius: 3,
                },
            ],
        },
        options: {
            responsive: true,
            plugins: {
                legend: { position: "top", labels: { color: "#8b949e" } }
            },
            scales: {
                x: {
                    ticks: { color: "#8b949e", maxTicksLimit: 10 },
                    grid:  { color: "rgba(48,54,61,0.8)" }
                },
                y: {
                    ticks: { color: "#8b949e", callback: v => "$" + v.toFixed(0) },
                    grid:  { color: "rgba(48,54,61,0.8)" }
                }
            }
        }
    });
}

function renderForecastGrid(predictions, currentPrice) {
    const grid = document.getElementById("forecastGrid");
    const today = new Date();
    grid.innerHTML = predictions.map((price, i) => {
        const d = new Date(today);
        d.setDate(d.getDate() + i + 1);
        const label = d.toLocaleDateString("en-US", { weekday: "short", month: "short", day: "numeric" });
        const diff = price - currentPrice;
        const cls = diff >= 0 ? "up" : "down";
        return `<div class="forecast-card">
      <span class="day">${label}</span>
      <span class="price">$${price.toFixed(2)}</span>
      <span class="diff ${cls}">${diff >= 0 ? "+" : ""}$${diff.toFixed(2)}</span>
    </div>`;
    }).join("");
}

function showError(msg) {
    const el = document.getElementById("error");
    el.textContent = msg;
    el.classList.remove("hidden");
}
function hideError() { document.getElementById("error").classList.add("hidden"); }
function showLoader(show) {
    document.getElementById("loader").classList.toggle("hidden", !show);
}
</script>
"""

In [7]:
# HTML

HTML = """
<!DOCTYPE html>
<html lang="en">
<head>
  <meta charset="UTF-8" />
  <meta name="viewport" content="width=device-width, initial-scale=1.0"/>
  <title>Stock Predictor</title>

    <link rel="preconnect" href="https://fonts.googleapis.com">
    <link rel="preconnect" href="https://fonts.gstatic.com" crossorigin>
    <link href="https://fonts.googleapis.com/css2?family=DM+Mono:wght@300;400;500&family=Syne:wght@400;500;600;700&display=swap" rel="stylesheet">

  <!-- Flask serves static files from /static/ automatically -->
  <!-- <link rel="stylesheet" href="{{ url_for('static', filename='style.css') }}"> -->
  <!-- <script src="{{ url_for('static', filename='app.js') }}"></script> -->

  {css}

</head>
<body>
  <div class="container">
    <header>
      <h1>📈 Stock Predictor</h1>
      <p>LSTM-powered price forecasting</p>
    </header>

    <div class="controls">
      <input id="ticker" type="text" placeholder="e.g. AAPL, TSLA" />
      <select id="days">
        <option value="7">7 days</option>
        <option value="14">14 days</option>
        <option value="30">30 days</option>
      </select>
      <button onclick="predict()">Predict</button>
    </div>

    <div id="error" class="error hidden"></div>
    <div id="loader" class="loader hidden">Training model...</div>

    <div id="results" class="hidden">
      <div class="metrics">
        <div class="card"><span>Current Price</span><strong id="currentPrice">—</strong></div>
        <div class="card"><span>7-Day Target</span><strong id="targetPrice">—</strong></div>
        <div class="card"><span>Ticker</span><strong id="tickerLabel">—</strong></div>
      </div>
      <canvas id="chart"></canvas>
      <div id="forecastGrid" class="forecast-grid"></div>
    </div>
  </div>

   <!-- <script src="{{ url_for('static', filename='app.js') }}"></script> -->
   {js}
</body>
</html>
""".format(css=CSS, js=JS)

In [8]:
warnings.filterwarnings("ignore")
LOOKBACK = 60

def fetch_data(ticker: str, period: str = "2y") -> pd.DataFrame:
    df = yf.download(ticker, period=period, progress=False)

    if df.empty:
        raise ValueError(f"No data found for ticker: {ticker}")

    # Fix for newer yfinance multi-level columns
    if isinstance(df.columns, pd.MultiIndex):
        df.columns = df.columns.get_level_values(0)

    return df[["Close"]]


def prepare_data(df: pd.DataFrame):
    """Scale and reshape data for LSTM input."""
    scaler = MinMaxScaler(feature_range=(0, 1))
    scaled = scaler.fit_transform(df.values)

    X, y = [], []
    for i in range(LOOKBACK, len(scaled)):
        X.append(scaled[i - LOOKBACK : i, 0])
        y.append(scaled[i, 0])

    X, y = np.array(X), np.array(y)
    X = X.reshape((X.shape[0], X.shape[1], 1))  # LSTM expects 3D input
    return X, y, scaler


def build_model(input_shape):
    """Build a simple LSTM model."""
    model = Sequential(
        [
            LSTM(64, return_sequences=True, input_shape=input_shape),
            Dropout(0.2),
            LSTM(64, return_sequences=False),
            Dropout(0.2),
            Dense(32, activation="relu"),
            Dense(1),
        ]
    )
    model.compile(optimizer="adam", loss="mean_squared_error")
    return model


def predict_next_days(ticker: str, days: int = 7) -> dict:
    """Full pipeline: fetch → train → predict."""
    df = fetch_data(ticker)
    X, y, scaler = prepare_data(df)

    # Train/test split (80/20)
    split = int(len(X) * 0.8)
    X_train, y_train = X[:split], y[:split]

    model = build_model((X.shape[1], 1))
    model.fit(X_train, y_train, epochs=20, batch_size=32, verbose=0)

    # Predict future days
    last_sequence = scaler.transform(df.values)[-LOOKBACK:]
    predictions = []

    for _ in range(days):
        seq = last_sequence.reshape(1, LOOKBACK, 1)
        pred_scaled = model.predict(seq, verbose=0)
        pred_price = scaler.inverse_transform(pred_scaled)[0][0]
        predictions.append(round(float(pred_price), 2))
        # Slide window: drop oldest, add new prediction
        last_sequence = np.append(last_sequence[1:], pred_scaled)

    # Historical closing prices (last 30 days for chart)
    history = df["Close"].tail(30).reset_index()
    history.columns = ["Date", "Close"]  # force clean column names
    history["Date"] = history["Date"].dt.strftime("%Y-%m-%d")
    history["Close"] = history["Close"].astype(float)

    return {
        "ticker": ticker.upper(),
        "current_price": round(float(df["Close"].iloc[-1]), 2),
        "predictions": predictions,
        "history": history.to_dict(orient="records"),
    }

In [9]:
app = Flask(__name__)

@app.after_request
def no_cache(response):
    response.headers["Cache-Control"] = "no-store, no-cache, must-revalidate"
    response.headers["Pragma"] = "no-cache"
    return response

@app.route("/")
def index():
    return HTML

@app.route("/api/predict")
def predict():
    ticker = request.args.get("ticker", "").strip().upper()
    days = int(request.args.get("days", 7))

    if not ticker:
        return jsonify({"error": "Ticker is required"}), 400
    if not (1 <= days <= 30):
        return jsonify({"error": "Days must be 1–30"}), 400

    try:
        result = predict_next_days(ticker, days)
        return jsonify(result)
    except ValueError as e:
        return jsonify({"error": str(e)}), 404
    except Exception as e:
        import traceback
        traceback.print_exc()
        return jsonify({"error": str(e)}), 500

In [10]:
# You need a free ngrok account — get your token from https://dashboard.ngrok.com
ngrok.set_auth_token(ngrok_token)

# Open a tunnel to Flask
public_url = ngrok.connect(5000)
print(f"🌐 Public URL: {public_url}")

# Run Flask in a background thread (so Colab doesn't freeze)
threading.Thread(target=lambda: app.run(port=5000)).start()

🌐 Public URL: NgrokTunnel: "https://canister-cheesy-spindle.ngrok-free.dev" -> "http://localhost:5000"
